# 05 — Your first classifier: nearest centroid

*Module 00 · Getting Started — notebook 5 of 11*

Everything so far has been preparation. Now you build a real classifier — by hand — and run it through the honest loop, end to end. This is the notebook the module has been building toward.

**Prerequisites:** 02 (the mean and Euclidean distance), 04 (the train/test split).

**What you'll be able to do**
- Build the **nearest-centroid** classifier by hand, and wrap it in the `fit` / `predict` API.
- Read its **decision boundary** and explain why it is a straight line.
- Run the honest loop — fit on train, evaluate on the held-out test set — and beat a baseline.
- State what the method **assumes** and where it would break.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestCentroid

from ml_course import colors, datasets, viz

np.random.seed(0)
viz.use_course_style()

X, y = datasets.penguins_xy()
FEATURES = datasets.PENGUINS_FEATURES
SPECIES_ORDER = ["Adelie", "Gentoo"]

# The same sealed split as notebook 04 (same seed, stratified).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

## Where we are

Two notebooks give us everything we need. Notebook 02 showed how to find the **centroid** (mean point) of a group of points, and how to measure the **distance** between two points. Notebook 04 split our penguins into a training set and a sealed test set. We now combine them into a classifier — and judge it the honest way.

## The idea: nearest centroid

The method is as plain as it sounds:

1. On the **training** penguins, compute the centroid (mean point) of each species.
2. To classify a new penguin, measure its distance to each centroid and assign it to the **nearest** one.

Two points summarize each species; a new penguin joins whichever it sits closer to. Let's compute those two points.

In [ ]:
centroids = X_train.groupby(y_train).mean()
species_list = list(centroids.index)

print("Training-set centroids (one point per species):")
print(centroids.round(2).to_string())

### Read the output

These two points — one per species, computed only from the training set — are the entire model. Computing them is what "fitting" means here: there is nothing else to learn. The Gentoo centroid sits up and to the right (longer flippers, larger bills), the Adélie centroid down and to the left, echoing what we saw in the scatter.

## The predict rule

To classify a penguin, we compute its Euclidean distance (notebook 02) to each centroid and pick the smaller. Let's do it explicitly for a couple of held-out penguins, showing both distances and the winner.

In [ ]:
def distances_to_centroids(point):
    """Euclidean distance from one penguin (a Series) to each species centroid."""
    return {s: float(np.sqrt(((point - centroids.loc[s]) ** 2).sum())) for s in species_list}


# one held-out penguin of each species, so we watch the rule resolve both ways
example_idx = [
    int(np.argmax((y_test == "Adelie").to_numpy())),
    int(np.argmax((y_test == "Gentoo").to_numpy())),
]
for i in example_idx:
    point = X_test.iloc[i]
    d = distances_to_centroids(point)
    nearest = min(d, key=d.get)
    print(f"test penguin: bill {point[FEATURES[0]]:.1f} mm, flipper {point[FEATURES[1]]:.0f} mm")
    for s, dist in d.items():
        print(f"    distance to {s} centroid: {dist:.1f}")
    print(f"    -> nearest centroid: {nearest}   (true species: {y_test.iloc[i]})\n")

### Read the output

For each penguin — here one of each species — we get two distances, one to each centroid, and assign it to the nearer one. Compare the chosen species with the true species printed alongside: the nearest centroid is the right call both times. That is the whole prediction rule, one penguin at a time.

## Wrap it as an estimator

Doing this one penguin at a time is good for understanding, but we want a reusable object. Every scikit-learn model shares the same two-method shape: **`fit(X, y)`** learns from training data, **`predict(X)`** labels new data. We wrap our rule in exactly that shape — which also lets us plot its decision boundary.

In [ ]:
class NearestCentroidByHand:
    """From-scratch nearest-centroid classifier.

    ``fit`` stores the class means; ``predict`` assigns each point to the nearest mean.
    """

    def fit(self, X, y):
        self.centroids_ = X.groupby(y).mean()
        self.classes_ = list(self.centroids_.index)
        self._C = self.centroids_.to_numpy()
        return self

    def predict(self, X):
        points = np.asarray(X, dtype=float)
        # distance from every point to every centroid, then take the nearest
        dists = np.sqrt(((points[:, None, :] - self._C[None, :, :]) ** 2).sum(axis=2))
        return np.array([self.classes_[i] for i in dists.argmin(axis=1)])


clf = NearestCentroidByHand().fit(X_train, y_train)
print("fitted centroids:")
print(clf.centroids_.round(2).to_string())

In [ ]:
fig = viz.plot_decision_boundary(clf, X_train, y_train, resolution=300)
ax = fig.axes[0]
for s in species_list:
    c = centroids.loc[s]
    ax.scatter(
        c[FEATURES[0]],
        c[FEATURES[1]],
        marker="X",
        s=320,
        color=colors.COLORS["text"],
        zorder=6,
    )
ax.set_title("Nearest-centroid decision boundary (X = centroids)")
plt.show()

### Read the figure

The two shaded regions are the model's territory: every point in a region is predicted as that species. The boundary between them is the set of points **equidistant** from the two centroids (marked `X`) — anything closer to the Gentoo centroid is called Gentoo, anything closer to the Adélie centroid, Adélie. Geometrically, that equidistant set is the **perpendicular bisector** of the segment joining the centroids: a straight line, exactly halfway. (It meets that segment at a true right angle in millimetre units; on these axes, whose two scales differ, it will not look like 90° on screen — a mismatch we resolve by standardizing features in notebook 11.) The model is two points and the line halfway between them.

## The honest loop

The model is fit on the training set. Now we run the rest of the loop the way notebook 04 insisted: **predict on the held-out test set** — penguins the centroids never saw — and measure the fraction it gets right, against the majority baseline. We also fit scikit-learn's own `NearestCentroid` to confirm our by-hand version agrees.

In [ ]:
test_pred = clf.predict(X_test)
test_score = (test_pred == y_test.to_numpy()).mean()
baseline = y_test.value_counts(normalize=True).max()

sklearn_clf = NearestCentroid().fit(X_train, y_train)
sklearn_score = (sklearn_clf.predict(X_test) == y_test.to_numpy()).mean()
agree = bool((test_pred == sklearn_clf.predict(X_test)).all())

print(f"nearest centroid (by hand) — fraction right on test: {test_score:.3f}")
print(f"majority baseline                                  : {baseline:.3f}")
print(f"scikit-learn NearestCentroid — fraction right       : {sklearn_score:.3f}")
print(f"by-hand and scikit-learn agree on every prediction  : {agree}")

### Read the output

On the sealed test set the nearest-centroid classifier gets **100%** right — far above the 55% majority baseline — and our by-hand version agrees with scikit-learn on every prediction. A perfect held-out score is a real, earned result: the test set was sealed, so this is honest generalization, not the memorizer's illusion from notebook 04.

Read it for what it is, though. These two species are unusually well separated on these two measurements, so even this very plain rule places every test penguin correctly. The lesson is that an honest, held-out score *can* be high — not that classification is always this easy. Later chapters meet harder problems where a straight line is not enough.

## What it assumes, where it breaks

A straight bisector is a strong assumption in disguise. Nearest-centroid effectively expects each class to be a roughly round cloud of similar size, balanced around its centre — then the halfway line is a sensible boundary. When a class is stretched out, curved, or far more spread than the other, the centroid no longer represents it well and the straight boundary misplaces points. (Our penguins happen to fit the friendly case.)

It also inherits notebook 02's caveat: distances mix bill millimetres with flipper millimetres, so the classifier is **scale-sensitive**. We standardize features and watch the effect in notebook 11.

## A score, not only a label

Right now the classifier answers only "Adélie" or "Gentoo." But it knows more than it says: a penguin much closer to one centroid is a confident call, while one near the boundary is a close one. That "how much closer" is a **score**, and turning labels into scores is what lets us draw ROC curves and choose thresholds — the subject of notebook 08.

## Your turn

1. Take a penguin at bill 44 mm, flipper 205 mm. Using the two training centroids printed above, compute its distance to each (by hand or in a new cell) and say which species nearest-centroid predicts.
2. If the Gentoo centroid moved further up and to the right, which way would the boundary shift — toward Adélie territory, or away from it?
3. Sketch, in words, a two-class dataset where nearest-centroid would do poorly, and say why.

Notebook 06 asks the question we have been postponing: is "fraction right" really a good way to judge a classifier?

## What you built

You built your first classifier, end to end, and judged it honestly:

- You computed class **centroids** and turned "nearest centroid" into predictions.
- You wrapped the rule in the **`fit` / `predict`** API that every scikit-learn model shares.
- You read its **decision boundary** — the perpendicular bisector — and matched scikit-learn's own result.
- You evaluated on held-out data, beat the baseline, and stated honestly what the method assumes and where it breaks.

**Vocabulary added:** nearest-centroid classifier, fit / predict (the estimator API), decision boundary, inductive bias, scale-sensitivity.

This is a genuine milestone — from "what is machine learning?" to a working, honestly evaluated classifier in five notebooks.

## References

- T. Hastie, R. Tibshirani, J. Friedman, *The Elements of Statistical Learning*, 2nd ed., Springer, 2009 — §4.3 (linear and centroid discriminants). DOI: 10.1007/978-0-387-84858-7
- R. Tibshirani, T. Hastie, B. Narasimhan, G. Chu (2002), *Diagnosis of multiple cancer types by shrunken centroids of gene expression*, PNAS 99(10):6567–6572. DOI: 10.1073/pnas.082099299
- scikit-learn developers, `sklearn.neighbors.NearestCentroid`.

---
Previous: **04 — Generalize, don't memorize** · Next: **06 — Is it any good? accuracy and a baseline**